# Ansible

In this notebook, we will use Ansible to configure our two virtual machines running Ubuntu Server 24.04 LTS we previously created using Terraform. On one virtual machine, we will install Docker, Gitea, and a Gitea Runner. On the other virtual machine, we will install one Kubernetes node. We will do this via **Ansible playbooks**.

Ansible is an automation language that can describe any IT environment, whether homelab or large scale infrastructure. It is easy to learn, and has beautiful code that reads like clear documentation. As an automation engine, Ansible ensures that your IT environment stays exactly as you describe it, no matter the complexity. Not only that, you can automate any command with Ansible to eliminate drudgery from your daily routine. Ansible gives you tooling to be more productive and solve problems that really matter.

**Ansible playbooks** are files that define a set of automation tasks to be executed on remote systems using Ansible. They are written in **YAML** and act like a blueprint for configuring servers, deploying applications, or managing infrastructure.

- Ansible Install:
  - [Installing Ansible on specific operating systems](https://docs.ansible.com/projects/ansible/latest/installation_guide/installation_distros.html)
- Ansible Documentation:
  - [Ansible community documentation](https://docs.ansible.com)
  - [Ansible playbooks](https://docs.ansible.com/projects/ansible/latest/playbook_guide/playbooks_intro.html#playbook-syntax)
  - [Working with playbooks](https://docs.ansible.com/projects/ansible/latest/playbook_guide/playbooks.html#working-with-playbooks) 
  - [Introduction to modules](https://docs.ansible.com/projects/ansible/latest/module_plugin_guide/modules_intro.html)
  - [How to build your inventory](https://docs.ansible.com/projects/ansible/latest/inventory_guide/intro_inventory.html)
  - [Ansible configuration settings](https://docs.ansible.com/projects/ansible/latest/reference_appendices/config.html) 
  - [YAML syntax](https://docs.ansible.com/projects/ansible/latest/reference_appendices/YAMLSyntax.html)
- Ansible modules and plugins
  - [Collection index](https://docs.ansible.com/projects/ansible/latest/collections/index.html)
  - [Indexes of all plugins](https://docs.ansible.com/projects/ansible/latest/collections/all_plugins.html)
  - [Indexes of all modules](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html)
    - [ansible-builtin](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html#ansible-builtin) 
      - [ansible-builtin-apt](https://docs.ansible.com/projects/ansible/latest/collections/ansible/builtin/apt_module.html#ansible-collections-ansible-builtin-apt-module) 

## Prerequisites

### Let's Create an Ansible Vault Password File
- Ansible uses something called a *vault* to store encrypted variables, i.e. sensitive information such as passwords.
- A *vault* is just a YAML file, where its contents has been encrypted by Ansible.
- We can define a password we want to use to encrypt and descrypt a YAML file.
  - We will use the password `student`.

Run the cell below to store the password `student` in the file `~/.ansible/.vault-pass.txt`

In [12]:
!echo "student" > ~/.ansible/.vault-pass.txt

## Show Ansible version

In [13]:
!ansible --version

ansible [core 2.16.3]
  config file = None
  configured module search path = ['/home/student/.ansible/plugins/modules', '/usr/share/ansible/plugins/modules']
  ansible python module location = /usr/lib/python3/dist-packages/ansible
  ansible collection location = /home/student/.ansible/collections:/usr/share/ansible/collections
  executable location = /usr/bin/ansible
  python version = 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0] (/usr/bin/python3)
  jinja version = 3.1.2
  libyaml = True


## Show Ansible help

- The command `ansible --help` shows help about ansible commands.
- The command `ansible-playbook --help` shows help about ansible playbook commands.
- The command `ansible-config --help` shows help about ansible configuration commands.
- The command `ansible-vault --help` shows help about ansible vault commands.
  - The command `ansible-vault [command] --help` shows additional help about a specific playbook command, e.g. `ansible-vault edit --help`.

In [14]:
!ansible --help
#!ansible-playbook --help
#!ansible-config --help
#!ansible-vault --help

usage: ansible [-h] [--version] [-v] [-b] [--become-method BECOME_METHOD]
               [--become-user BECOME_USER]
               [-K | --become-password-file BECOME_PASSWORD_FILE]
               [-i INVENTORY] [--list-hosts] [-l SUBSET] [-P POLL_INTERVAL]
               [-B SECONDS] [-o] [-t TREE] [--private-key PRIVATE_KEY_FILE]
               [-u REMOTE_USER] [-c CONNECTION] [-T TIMEOUT]
               [--ssh-common-args SSH_COMMON_ARGS]
               [--sftp-extra-args SFTP_EXTRA_ARGS]
               [--scp-extra-args SCP_EXTRA_ARGS]
               [--ssh-extra-args SSH_EXTRA_ARGS]
               [-k | --connection-password-file CONNECTION_PASSWORD_FILE] [-C]
               [-D] [-e EXTRA_VARS] [--vault-id VAULT_IDS]
               [-J | --vault-password-file VAULT_PASSWORD_FILES] [-f FORKS]
               [-M MODULE_PATH] [--playbook-dir BASEDIR]
               [--task-timeout TASK_TIMEOUT] [-a MODULE_ARGS] [-m MODULE_NAME]
               pattern

Define and run a single task '

## Ansible Configuration Files

An Ansible configuration file has the file extension `.cfg`, and is used to configure basic settings for an Ansible project.
- We can view the contents of our configuration file `ansible.cfg` (in the `ansible` folder) using the command `ansible-config view`.

In the configuration file, we place default settings for our Ansible project under the [defaults] section.
- `inventory = inventory.ini` specifies which *inventory* file Ansible should use by default.
- `private_key_file = ~/.ssh/id_ed25519_ansible` specifies which SSH private key file to use when Ansible connects to hosts (for the `ansible` user).
- `interpreter_python = auto_silent` specifies how Ansible should locate the Python interpreter on remote hosts (Ansible uses Python on remote hosts).
- `host_key_checking = False` specifies if SSH host key verifiction should be used (we set this to `False` to avoid prompts when connecting to hosts).
- `vault_password_file = ~/.ansible/.vault-pass.txt` specifies the path to a file containing the Ansible Vault password (used for encrypting/decrypting variables).
- `remote_user = ansible` specifies the default SSH user Ansible will use when connecting to hosts (i.e. the `ansible` user we have installed on our VMs).
- `allow_world_readable_tmpfiles = True` specifies if temporary files created by Ansible are world-readable (i.e. readable by any user).
- `remote_tmp = /tmp/.ansible-${USER}/tmp` specifies the directory on remote hosts where Ansible will store temporary files (`${USER}` gets replaced with `ansible`).

In [15]:
%%bash

cd ansible
ansible-config view

[defaults]
inventory = inventory.ini
private_key_file = ~/.ssh/id_ed25519_ansible
interpreter_python = auto_silent
host_key_checking = False
vault_password_file = ~/.ansible/.vault-pass.txt
remote_user = ansible
allow_world_readable_tmpfiles = True
remote_tmp = /tmp/.ansible-${USER}/tmp


## Ansible Inventory Files

An Ansible inventory file has the file extension `.ini`, and is used to specify the hosts (the computers/VMs) for an Ansible project.
- Let's view the contents of our inventory file `inventory.ini` (in the `ansible` folder) below.

In the inventory file, we can group hosts (computers/VMs) under sections `[section_name]`, where `section_name` is the name of the section.
- Under the `[gitea]` section, we have specified a single host.
  - The hostname `u24s01` and IP address `ansible_host=192.168.122.99` of one of our Ubuntu Server 24.04 LTS VMs.
  - We will install Docker, Gitea, and a Gitea Runner on this VM.
- Under the `[k8s]` section, we have specified a single host.
  - The hostname `u24s02` and IP address `ansible_host=192.168.122.98` of our other Ubuntu Server 24.04 LTS VM.
  - We will install Kubernetes on this VM.

In our Ansible Playbooks, we can refer to a section to specify we want to configure all hosts under that section (in our case we only have one host under each section).

In [16]:
# !type ansible\inventory.ini # use this on Windows
!cat ansible/inventory.ini # use this on Linux/Mac

[gitea]
u24s01 ansible_host=192.168.122.99

[k8s]
u24s02 ansible_host=192.168.122.98

; [all:vars]
; ansible_connection=ssh
; ansible_port=22
; ansible_user=ansible
; ansible_ssh_private_key_file=~/.ssh/id_ed25519_ansible


## Ansible Variables

Ansible variables are key-value pairs that we can define, which are later used in our **Ansible playbooks**.
- We place Ansible variables in YAML files under the special folder structure `group_vars/all`.
- In our case, we will place our variables in two files:
  - `vars.yml` will contain **unencrypted** variables (this is where we place non-sensitive variables).
  - `vault.yml` will contain **encrypted** variables (this is where we place sensitive variables such as passwords).
- A variable file can be encrypted using `ansible-vault`, where Ansible will automatically decrypt encrypted variables when used in an Ansible Playbook.
  - It will use the password in the file `~/.ansible/.vault-pass.txt` to encrypt/decrypt the file `vault.yml`.

Let's view the contents of the file `vars.yml` below.
- We see that it contains a two variables:
  - A variable with the name `gitea_site_url` and value `http://192.168.122.99:3000`.
    - The value contains the URL to the Gitea Web UI (it will be available after we have installed Gitea).
  - A variable with the name `gitea_site_ip` and value `192.168.122.99:3000`.
    - The value contains the IP address to the Gitea Web UI (it will be available after we have installed Gitea).

In [17]:
# !type ansible\group_vars\all\vars.yml # use this on Windows
!cat ansible/group_vars/all/vars.yml # use this on Linux/Mac

gitea_site_url: http://192.168.122.99:3000
gitea_site_ip: 192.168.122.99:3000

Let's also view the contents of the file `vault.yml` below.
- Since the file `vault.yml` has been encrypted with `ansible-vault`, we need to use the command `ansible-vault view group_vars/all/vault.yml` to view its contents.
  - It will use the password we previously stored in the file `~/.ansible/.vault-pass.txt` to decrypt the contents of `vault.yml`.
- We see that it contains two variables:
  - A variable with the name `password` and value `"student"`
    - This is the password for the Gitea admin user we will configure in one of our Ansible Playbooks.
  - `gitea_admin_token: "91d1fbab768f9e668748c634245aca08a4f76496"`
    - This contains a Gitea Token we will use to communicate with Gitea via its REST API (we will create a new token and update this variable after installing Gitea).

In [18]:
%%bash

cd ansible
ansible-vault view group_vars/all/vault.yml

password: "student"
gitea_admin_token: "6ea8ab3015bdf889fb82b4920263c4d72a8ccfd3"


## Ansible Playbooks

[Ansible playbooks](https://docs.ansible.com/projects/ansible/latest/playbook_guide/playbooks_intro.html#playbook-syntax) are files that define a set of automation tasks to be executed on remote systems using Ansible. They are written in [YAML](https://docs.ansible.com/projects/ansible/latest/reference_appendices/YAMLSyntax.html) and act like a blueprint for configuring servers, deploying applications, or managing infrastructure (see [working with playbooks](https://docs.ansible.com/projects/ansible/latest/playbook_guide/playbooks.html#working-with-playbooks)).

Ansible Playbooks use various [collections of modules](https://docs.ansible.com/projects/ansible/latest/collections/index.html) (see [introduction to modules](https://docs.ansible.com/projects/ansible/latest/module_plugin_guide/modules_intro.html)) to perform various tasks on remote systems. For example, the [ansible.builtin](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html#ansible-builtin) collection contains the package management module [apt](https://docs.ansible.com/projects/ansible/latest/collections/ansible/builtin/apt_module.html#ansible-collections-ansible-builtin-apt-module) for running the `apt` command on Debian-based Linux distributions, e.g. Ubuntu Server 24.04 LTS, to install packages. We won't cover modules in detail, but Ansible's [indexes of all modules](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html) provides documentation for all modules.

Let's view the contents of the Ansible Playbook file `install-docker.yml` (in the folder `ansible/playbooks`) below.

- Each Ansible Playbook can contain one or more *plays*, each with a unique `name`, where each *play* is a YAML object, specified as a YAML array element. In `install-docker.yml`, we have a single `play`:
  - The initial `-` defines a new YAML array element.
  - `name: Install Docker Engine on Ubuntu / Ubuntu-like distros` gives this *play* a name.
  - `hosts: gitea` specifies the hosts that will be configure by this play.
    - `gitea` is the name of one of our sections in the file `inventory.ini`.
      - The secion has one host specified: `u24s01 ansible_host=192.168.122.99`.
  - `become: true` means Ansbile will become a superuser inside the host (i.e. issue commands with `sudo` inside the Ubuntu Server 24.04 LTS VMs).

    ```yaml
    - name: Install Docker Engine on Ubuntu / Ubuntu-like distros
      hosts: gitea
      become: true
    ```
  
  - Next, there is an optional `vars` section that defines two variables `docker_users` and `docker_insecure_registries` (both are arrays).
    - The construct `"{{ gitea_site_url }}"` reads the value of a variable defined in either the `vars.yml` file or the `vault.yml` file (in this case, the value of the variable `gitea_site_url` defined in `vars.yml`).
    - We also see that we can create line comments with the hash character `#`

    ```yaml
    vars:
    # Add any local users you want to run docker without sudo
    docker_users:
      - student
      - ansible
    
    docker_insecure_registries:
      - "{{ gitea_site_url }}"
    ```

  - The `vars` section is followed by an optional `pre_tasks` section, where we can define `tasks` we want to be run before any other `tasks`.
    - We see that each `task`:
      - Is specified as a YAML array element (`-`).
      - Has a `name`.
      - Uses an Ansible [module](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html) (in this case [ansible.builtin.set_fact](https://docs.ansible.com/projects/ansible/latest/collections/ansible/builtin/set_fact_module.html#ansible-collections-ansible-builtin-set-fact-module)) to perform a specific action (available property settings depend on the specific module).

    ```yaml
    pre_tasks:
      - name: Map architecture and Ubuntu codename
        ansible.builtin.set_fact:
          docker_arch: >-
            {{ {
                'x86_64':'amd64',
                'aarch64':'arm64',
                'armv7l':'armhf',
                'armv6l':'armhf'
              }[ansible_facts['architecture']] | default('amd64') }}
          ubuntu_codename: "{{ ansible_facts['lsb']['codename'] | default(ansible_facts['distribution_release']) }}"
    ```

  - The `pre_tasks` section is followed by an optional `handlers` section, where we can define *handlers*, which are just `tasks` that can be called by other `tasks`.
    - Once again, a task is defined as an array element, with a name and a [module](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html) (in this case [ansible.builtin.service](https://docs.ansible.com/projects/ansible/latest/collections/ansible/builtin/service_module.html#ansible-collections-ansible-builtin-service-module)) with module-specific property settings.

    ```yaml
    handlers:
    - name: Restart docker
      ansible.builtin.service:
        name: docker
        state: restarted
    ```
  
  - The `handlers` section is followed by a mandatory `tasks` section, in which we specify the main `tasks` we want to execute inside the hosts.
    - Once again, a task is defined as an array element, with a name and a [module](https://docs.ansible.com/projects/ansible/latest/collections/index_module.html) (in this case [ansible.builtin.apt](https://docs.ansible.com/projects/ansible/latest/collections/ansible/builtin/apt_module.html#ansible-collections-ansible-builtin-apt-module)) with module-specific property settings, e.g.:

    ```yaml
    - name: Install prerequisite packages
        ansible.builtin.apt:
          name:
            - ca-certificates
            - curl
            - gnupg
            - lsb-release
          state: present
          update_cache: true
    ```
- This Ansible Playbook installs Docker on one of our Ubuntu Server 24.04 LTS VMs.

In [19]:
# !type ansible\playbooks\install-docker.yml # use this on Windows
!cat ansible/playbooks/install-docker.yml # use this on Linux/Mac

- name: Install Docker Engine on Ubuntu / Ubuntu-like distros
  hosts: gitea
  become: true

  vars:
    # Add any local users you want to run docker without sudo
    docker_users:
      - student
      - ansible
    
    docker_insecure_registries:
      - "{{ gitea_site_ip }}"

  pre_tasks:
    - name: Map architecture and Ubuntu codename
      ansible.builtin.set_fact:
        docker_arch: >-
          {{ {
              'x86_64':'amd64',
              'aarch64':'arm64',
              'armv7l':'armhf',
              'armv6l':'armhf'
            }[ansible_facts['architecture']] | default('amd64') }}
        ubuntu_codename: "{{ ansible_facts['lsb']['codename'] | default(ansible_facts['distribution_release']) }}"

  handlers:
    - name: Restart docker
      ansible.builtin.service:
        name: docker
        state: restarted
  
  tasks:
    - name: Remove legacy Docker packages (safe if absent)
      ansible.builtin.apt:
        name:
          - docker
          - docker-engine
  

## Ansible-Playbook

The command `ansible-playbook playbook`, where `playbook` is the path to a Playbook YAML file, runs (executes) the Playbook.

### Let's install Docker on one of our Ubuntu Server 24.04 LTS VMs

- To install Docker, we'll run the playbook `playbooks/install-docker.yml`.
  - In the Playbook we have the setting below:
    
    ```YAML
    hosts: gitea
    ```
  - In our inventory `inventory.ini` we have defined the `[gitea]` section as below:
    
    ```YAML
    [gitea]
    u24s01 ansible_host=192.168.122.99
    ```
  
  - This means the Playbook tasks are run (executed) against the VM with IP address `192.168.122.99` (which happens to have hostname `u24s01`).

In [20]:
%%bash

cd ansible
ansible-playbook playbooks/install-docker.yml


PLAY [Install Docker Engine on Ubuntu / Ubuntu-like distros] *******************

TASK [Gathering Facts] *********************************************************
ok: [u24s01]

TASK [Map architecture and Ubuntu codename] ************************************
ok: [u24s01]

TASK [Remove legacy Docker packages (safe if absent)] **************************
ok: [u24s01]

TASK [Install prerequisite packages] *******************************************
ok: [u24s01]

TASK [Ensure /etc/apt/keyrings exists] *****************************************
ok: [u24s01]

TASK [Download Docker GPG key (ASCII)] *****************************************
changed: [u24s01]

TASK [Convert ASCII key to keyring] ********************************************
changed: [u24s01]

TASK [Ensure keyring is world-readable] ****************************************
ok: [u24s01]

TASK [Add Docker APT repository] ***********************************************
changed: [u24s01]

TASK [Update apt cache] ***********************

### Let's install Gitea on the same Ubuntu Server 24.04 LTS VM
- To install Gitea, we'll run the playbook `playbooks/install-gitea.yml`.
  - In the Playbook we have the setting below:
    
    ```YAML
    hosts: gitea
    ```
  - In our inventory `inventory.ini` we have defined the `[gitea]` section as below:
    
    ```YAML
    [gitea]
    u24s01 ansible_host=192.168.122.99
    ```
  
  - This means the Playbook tasks are run (executed) against the VM with IP address `192.168.122.99` (which happens to have hostname `u24s01`).

In [21]:
%%bash

cd ansible
ansible-playbook playbooks/install-gitea.yml


PLAY [Install and configure Gitea with admin user (no GUI)] ********************

TASK [Gathering Facts] *********************************************************
ok: [u24s01]

TASK [Ensure dependencies] *****************************************************
ok: [u24s01]

TASK [Ensure gitea group exists] ***********************************************
changed: [u24s01]

TASK [Create gitea user] *******************************************************
changed: [u24s01]

TASK [Create required directories] *********************************************
changed: [u24s01] => (item={'path': '/var/lib/gitea'})
changed: [u24s01] => (item={'path': '/var/lib/gitea/custom'})
changed: [u24s01] => (item={'path': '/var/lib/gitea/custom/conf'})
changed: [u24s01] => (item={'path': '/var/lib/gitea/data'})
changed: [u24s01] => (item={'path': '/var/lib/gitea/indexers'})
changed: [u24s01] => (item={'path': '/var/lib/gitea/log'})

TASK [Download Gitea binary] *************************************************

[WARNING]: Using world-readable permissions for temporary files Ansible needs
to create when becoming an unprivileged user. This may be insecure. For
information on securing this, see https://docs.ansible.com/ansible-
core/2.16/playbook_guide/playbooks_privilege_escalation.html#risks-of-becoming-
an-unprivileged-user
[WARNING]: Module remote_tmp /tmp/.ansible-git/tmp did not exist and was
created with a mode of 0700, this may cause issues when running as another
user. To avoid this, create the remote_tmp dir with the correct permissions
manually


changed: [u24s01]

TASK [Generate INTERNAL_TOKEN] *************************************************
changed: [u24s01]

TASK [Generate LFS_JWT_SECRET] *************************************************
changed: [u24s01]

TASK [Generate OAuth2 JWT_SECRET] **********************************************
changed: [u24s01]

TASK [Write custom app.ini] ****************************************************
changed: [u24s01]

TASK [Remove legacy /etc/gitea/app.ini if present (avoid confusion)] ***********
ok: [u24s01]

TASK [Install/refresh systemd unit] ********************************************
changed: [u24s01]

TASK [Run DB migrations] *******************************************************
changed: [u24s01]

TASK [Enable and start Gitea] **************************************************
changed: [u24s01]

TASK [Wait until Gitea HTTP is up] *********************************************
ok: [u24s01]

TASK [List existing users (idempotency check)] *********************************
ok: [u24s0

[WARNING]: Could not match supplied host pattern, ignoring: servers



PLAY [Create Gitea Admin Token (API) and Store in Ansible Vault] ***************
skipping: no hosts matched

PLAY RECAP *********************************************************************
u24s01                     : ok=20   changed=14   unreachable=0    failed=0    skipped=0    rescued=0    ignored=0   



### Creating a Gitea Admin Token

Now that Gitea is installed, we'll create a token, for the admin user, we can use to communicate with Gitea via its REST API.
- Visit http://192.168.122.99:3000
- Click `Sign In`
- Login with:
  - Username: `student`
  - Password: `student`
- Click you profile avatar (`Profiles and Settings`) in the top right.
- Choose `Settings -> Applications`
- Under `Generate New Token`, enter:
  - Token Name: `admin`
  - Repository and Organization Access: `All (public, private, and limited)`
  - Under `Selected token permissions limit authorization only to the corresponding API routes. Read the documentation for more information.`
    - Set all to `Read and Write`
- Click the `Generate Token` button.
- Copy and save the token (which looks something like this: `28587c1cd851b1ad4fb92c4bbe81986c91b629e3`).

### Storing the Gitea Admin Token in Ansible Vault

Let's store the Gitea Admin Token in Ansible Vault (`group_vars/all/vault.yml`) so we can use it in the next Ansible Playbook.

1. Open the file `group_vars/all/vault.yml` in VSCode.
   - Notice the file is encrypted.

2. Open a terminal.

3. Decrypt the file with the command below.
   - Notice the file is now decrypted in VSCode. 
   
   ```bash
   ansible-vault decrypt group_vars/all/vault.yml
   ```

4. Edit the file and make sure you replace the value for `gitea_admin_token` with YOUR token from above.

   ```bash
   gitea_admin_token: "28587c1cd851b1ad4fb92c4bbe81986c91b629e3"
   gitea_site_url: "http://192.168.122.99:3000"
   ````

5. Encrypt the file again with the command:
   - Notice the file is now encrypted again. 

   ```bash
   ansible-vault encrypt group_vars/all/vault.yml
   ```

**Note**
- You can edit an encrypted file directly with the command below.
  - This will use your default editor (`vim` on a university VM).
   
   ```bash
   ansible-vault edit group_vars/all/vault.yml
   ```

- You can view the contents of a file with the command:
  - This will list the file's contents (encrypted or decrypted).

  ```bash
  ansible-vault view group_vars/all/vault.yml
  ```

### Let's install a Gitea Runner on the same Ubuntu Server 24.04 LTS VM
- To install a Gitea Runner, we'll run the playbook `playbooks/install-gitea-runners.yml`.
  - In the Playbook we have the setting below:
    
    ```YAML
    hosts: gitea
    ```
    
  - In the Playbook's first `task`, we also have the setting below:
    - This reads the value of the `gitea_admin_token` variable from `ansible/group_vars/all/vault.yml`. 

    ```bash
    headers:
      Authorization: 'token {{ gitea_admin_token }}'
    ```
  
  - In our inventory `inventory.ini` we have defined the `[gitea]` section as below:
    
    ```YAML
    [gitea]
    u24s01 ansible_host=192.168.122.99
    ```
  
  - This means the Playbook tasks are run (executed) against the VM with IP address `192.168.122.99` (which happens to have hostname `u24s01`).

In [22]:
%%bash

cd ansible
ansible-playbook playbooks/install-gitea-runners.yml


PLAY [Create Gitea Admin Token (API) & Fetch Runner Registration Token] ********

TASK [Gathering Facts] *********************************************************
ok: [u24s01]

TASK [Ensure shared Ansible temp dir exists] ***********************************
changed: [u24s01]

TASK [Fetch global runner registration token (admin-only)] *********************
ok: [u24s01]

TASK [Save runner registration token for runner play] **************************
ok: [u24s01]

PLAY [Register and Start Gitea Runners] ****************************************

TASK [Gathering Facts] *********************************************************
ok: [u24s01]

TASK [Create runner base dirs] *************************************************
changed: [u24s01] => (item=u24s01-runner-01)

TASK [Download act_runner binary if needed (per runner dir)] *******************
changed: [u24s01] => (item=u24s01-runner-01)

TASK [Register each runner (non-interactive)] **********************************
changed: [u24s01] =>

### Let's install Kubernetes on our other Ubuntu Server 24.04 LTS VM
- To install Kubernetes, we'll run the playbook `playbooks/install-k8s.yml`.
  - In the Playbook we have the setting below:
    
    ```YAML
    hosts: k8s
    ```
  - In our inventory `inventory.ini` we have defined the `[k8s]` section as below:
    
    ```YAML
    [k8s]
    u24s02 ansible_host=192.168.122.98
    ```
  
  - This means the Playbook tasks are run (executed) against the VM with IP address `192.168.122.98` (which happens to have hostname `u24s02`).

**Note**
- The Playbook will store the Kubernetes configuration file in: `~/.kube/config-u24s02`
- To access the Kubernetes cluster, rename the file to `~/.kube/config`, or set your terminal's environment variable `KUBECONFIG` to `~/.kube/config-u24s02`.
  - On a Linux machine (e.g. if you are using one of the university's VMs), you can do this with the commmand below:

    ```bash
    export KUBECONFIG=/home/student/.kube/config-u24s02
    ```

In [23]:
%%bash

cd ansible
ansible-playbook playbooks/install-k8s.yml


PLAY [Install single-node Kubernetes on Ubuntu 24.04] **************************

TASK [Gathering Facts] *********************************************************
ok: [u24s02]

TASK [Update apt cache] ********************************************************
ok: [u24s02]

TASK [Install base packages] ***************************************************
changed: [u24s02]

TASK [Disable swap immediately] ************************************************
changed: [u24s02]

TASK [Disable swap in /etc/fstab] **********************************************
changed: [u24s02]

TASK [Load overlay kernel module] **********************************************
ok: [u24s02]

TASK [Load br_netfilter kernel module] *****************************************
ok: [u24s02]

TASK [Persist required kernel modules] *****************************************
changed: [u24s02]

TASK [Set Kubernetes sysctl params] ********************************************
changed: [u24s02]

TASK [Apply sysctl params] **********

### Let's Base64-encode the Kubernetes Configuration File
- Later when we access the Kubernetes cluster via Gitea Actions, we will need a Base64-encoded version of the Kubernetes configuration file.
- Run the cell below to:
  - Read the contents of the configuration file: `/home/student/.kube/config-u24s02`
  - Base64-encode its contents (now in RAM).
  - Store the Base64-encoded contents (in RAM) into the file: `/home/student/.kube/config-u24s02-base64`

**Note**

- On Windows, before running the cell below:
  - Download Base64 from here: https://www.di-mgt.com.au/base64-for-windows.html
    - Here's a direct link: https://www.di-mgt.com.au/src/base64-1.2.0.zip
  - Place the binary `Base64.exe` in the folder `workshop5/01_Infrastructure_as_code`.

In [24]:
!base64 /home/student/.kube/config-u24s02 > /home/student/.kube/config-u24s02-base64

### Let's Install `Metallb` into the Kubernetes Cluster
- Run the two cells below to:
  - Install `Metallb` Helm Chart into the Kubernetes Cluster using Helm.
  - Configure `Metallb` to hand out public IP addressed in the range `192.168.122.100` to `192.168.122.110`.

**Note:** Wait 30-60 seconds before running the seconds cell, so that MetalLB has finished installing.

In [25]:
%%bash

export KUBECONFIG=/home/student/.kube/config-u24s02
helm repo add metallb https://metallb.github.io/metallb
helm repo update
helm upgrade --install metallb metallb/metallb --create-namespace --namespace metallb-system

"metallb" has been added to your repositories
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "metrics-server" chart repository
...Successfully got an update from the "metallb" chart repository
Update Complete. ⎈Happy Helming!⎈
Release "metallb" does not exist. Installing it now.
NAME: metallb
LAST DEPLOYED: Thu Mar 19 20:27:36 2026
NAMESPACE: metallb-system
STATUS: deployed
REVISION: 1
TEST SUITE: None
NOTES:
MetalLB is now running in the cluster.

Now you can configure it via its CRs. Please refer to the metallb official docs
on how to use the CRs.


In [27]:
%%bash

export KUBECONFIG=/home/student/.kube/config-u24s02
kubectl apply -f metallb-config.yaml

ipaddresspool.metallb.io/default-pool created
l2advertisement.metallb.io/default-l2 created


### List all the pods in the Kubenetes cluster
- Notice Metallb's `speaker` pod (in namespace `metallb-system`) has an IP address on subnet `192.168.122.0/24`.
  - This is why we configured `Metallb` to hand out public IP addressed in the range `192.168.122.100` to `192.168.122.110`.

In [29]:
%%bash

export KUBECONFIG=/home/student/.kube/config-u24s02
kubectl get pods -A -o wide

NAMESPACE         NAME                                       READY   STATUS    RESTARTS   AGE     IP                NODE     NOMINATED NODE   READINESS GATES
calico-system     calico-apiserver-69fc465d8d-hlrbf          1/1     Running   0          3m21s   192.168.197.194   u24s02   <none>           <none>
calico-system     calico-apiserver-69fc465d8d-tmlp6          1/1     Running   0          3m21s   192.168.197.197   u24s02   <none>           <none>
calico-system     calico-kube-controllers-7d5cfcd887-4zcks   1/1     Running   0          3m20s   192.168.197.196   u24s02   <none>           <none>
calico-system     calico-node-5frst                          1/1     Running   0          3m20s   192.168.122.98    u24s02   <none>           <none>
calico-system     calico-typha-69d5f47949-nkrk4              1/1     Running   0          3m21s   192.168.122.98    u24s02   <none>           <none>
calico-system     csi-node-driver-br5sm                      2/2     Running   0          3m20s  

## Conclusion

We have now configured our two Ubuntu Server 24.04 LTS VMs.
- The VM with hostname `u24s01` and IP address `192.168.122.99` is running:
  - Docker Engine.
  - Gitea.
  - A Gitea Runner (in a Docker container) that we have registered with Gitea.
- The VM with hostname `u24s02` and IP address `192.168.122.98` is running:
  - Kubernetes (a single-node Kubernetes cluster).
  - Metallb inside the Kubernetes cluster.